**Note**: Please switch to GPU for this notebook (e.g. T4)

# Get DonkeyCar scripts

Clone directory

In [ ]:
!git clone https://github.com/UPenn-AI-X/donkeycar-bookworm /content/projects/donkeycar
%cd /content/projects/donkeycar
!git checkout main

Cloning into '/content/projects/donkeycar'...
remote: Enumerating objects: 13348, done.
remote: Total 13348 (delta 0), reused 0 (delta 0), pack-reused 13348 (from 1)
Receiving objects: 100% (13348/13348), 88.80 MiB | 24.04 MiB/s, done.
Resolving deltas: 100% (8717/8717), done.
/content/projects/donkeycar
Already on 'main'
Your branch is up to date with 'origin/main'.


Install dependencies

In [ ]:
# !sed -i 's/<3.12/<3.13/' setup.cfg # relax python version requirement
!sed -i 's/numpy>=1.23,<2.0/numpy>=1.23/' setup.cfg
!pip install -e . --ignore-requires-python
!pip install tf-keras

Obtaining file:///content/projects/donkeycar
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.6 MB/s  0:00:00
  Building editable for donkeycar (pyproject.toml) ... done
  Created wheel for donkeycar: filename=donkeycar-5.3.dev1-0.editable-py3-none-any.whl size=8472 sha256=f867a0ba7c411bba39d1fc02eade3399aa79658df8794897ef24caeb6453b084
  Stored in directory: /tmp/pip-ephem-wheel-cache-2icir4a5/wheels/64/da/d2/b4d028290c06ec30ade257411af2283589234309649ac34474
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=d4b905066796aa7081e29cda08372ecf4afa17d50769035090246534f99c5400
  Stored in directory: /root/

Create `mycar`

In [ ]:
!donkey createcar --path /content/mycar

________             ______                   _________              
___  __ \_______________  /___________  __    __  ____/_____ ________
__  / / /  __ \_  __ \_  //_/  _ \_  / / /    _  /    _  __ `/_  ___/
_  /_/ // /_/ /  / / /  ,<  /  __/  /_/ /     / /___  / /_/ /_  /    
/_____/ \____//_/ /_//_/|_| \___/_\__, /      \____/  \__,_/ /_/     
                                 /____/                              

using donkey v5.3.dev1 ...
Creating car folder: /content/mycar
making dir  /content/mycar
Creating data & model folders.
making dir  /content/mycar/models
making dir  /content/mycar/data
making dir  /content/mycar/logs
Copying car application template: complete
Copying car config defaults. Adjust these before starting your car.
Copying train script. Adjust these before starting your car.
Copying calibrate script. Adjust these before starting your car.
Copying my car config overrides
Donkey setup complete.


# Train model

First, upload zipped files & unzip in colab

In [ ]:
from google.colab import files
import zipfile
import os
from PIL import Image
import glob, re

uploaded = files.upload()
zip_path = next(iter(uploaded.keys()))
os.makedirs("/content/mycar/data", exist_ok=True)

# unzip, stripping top level folder
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    names = [n for n in zip_ref.namelist() if n and not n.startswith("__MACOSX")]
    # Detect if everything shares a single top-level folder
    top_levels = {n.split("/", 1)[0] for n in names}
    strip_prefix = len(top_levels) == 1 and any(n.startswith(next(iter(top_levels)) + "/") for n in names)
    prefix = next(iter(top_levels)) + "/" if strip_prefix else ""

    for member in names:
        if not member.startswith(prefix):
            continue
        rel = member[len(prefix):]
        if not rel:
            continue
        target_path = os.path.join("/content/mycar/data", rel)
        if member.endswith("/"):
            os.makedirs(target_path, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target_path), exist_ok=True)
            with zip_ref.open(member) as source, open(target_path, "wb") as target:
                target.write(source.read())

print(f"Extracted {zip_path} to /content/mycar/data")

# Find any image in the extracted data
image_files = glob.glob("/content/mycar/data/images/*.jpg")
if not image_files:
    raise RuntimeError("No images found in /content/mycar/data/images/. Check the zip structure.")

with Image.open(image_files[0]) as img:
    img_w, img_h = img.size
print(f"Detected image size: {img_w}x{img_h}")

# Patch config.py to match the actual data
config_path = "/content/mycar/config.py"
with open(config_path) as f:
    text = f.read()
text = re.sub(r'^IMAGE_W\s*=\s*\d+', f'IMAGE_W = {img_w}', text, flags=re.MULTILINE)
text = re.sub(r'^IMAGE_H\s*=\s*\d+', f'IMAGE_H = {img_h}', text, flags=re.MULTILINE)
with open(config_path, "w") as f:
    f.write(text)
print(f"Updated config.py: IMAGE_W={img_w}, IMAGE_H={img_h}")


Saving data_6-26_15-00.zip to data_6-26_15-00.zip
Extracted data_6-26_15-00.zip to /content/mycar/data
Detected image size: 240x80
Updated config.py: IMAGE_W=240, IMAGE_H=80


Train model

In [ ]:
%cd /content/mycar
!TF_USE_LEGACY_KERAS=1 donkey train --tub /content/mycar/data --model /content/mycar/models/mypilot.h5 --type linear

/content/mycar
________             ______                   _________              
___  __ \_______________  /___________  __    __  ____/_____ ________
__  / / /  __ \_  __ \_  //_/  _ \_  / / /    _  /    _  __ `/_  ___/
_  /_/ // /_/ /  / / /  ,<  /  __/  /_/ /     / /___  / /_/ /_  /    
/_____/ \____//_/ /_//_/|_| \___/_\__, /      \____/  \__,_/ /_/     
                                 /____/                              

using donkey v5.3.dev1 ...
INFO:donkeycar.config:loading config file: ./config.py
INFO:donkeycar.config:loading personal config over-rides from ./myconfig.py
I0000 00:00:1794257896.111403    2584 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:donkeycar.utils:get_model_by_type: model type 